In [15]:
import os
import shutil
import stat
import subprocess
from pathlib import Path

from google.colab import drive, userdata

In [17]:
import os
import shutil
import stat
import subprocess
from pathlib import Path

from google.colab import drive, userdata

"""
Google Colab -> GitHub data synchronization script

Purpose
-------
Mount Google Drive, copy/synchronize:
    /content/drive/MyDrive/ColabNotebooks/DataForReputation
to a folder in a GitHub repository, commit the changes, and push them.

Before running in Colab:
1. Create a GitHub fine-grained Personal Access Token (PAT).
2. Store it in Colab Secrets as: GITHUB_TOKEN
3. Edit GITHUB_USERNAME and GITHUB_REPOSITORY below.
4. Set PUBLIC_DATA_CONFIRMED = True only after confirming the source folder
   contains no secrets, restricted/private data, or files you cannot publish.
"""




# ============================================================
# USER CONFIGURATION
# ============================================================

GITHUB_USERNAME = "alinaantonov"
GITHUB_REPOSITORY = "DataForReputation"
BRANCH = "main"
GIT_USER_EMAIL = "alina_antonov@yahoo.com"

SOURCE_DIR = Path("/content/drive/MyDrive/ColabNotebooks/DataForReputation")

# Destination inside the GitHub repository.
# Change to "DataForReputation" if you want it at the repository root.
GITHUB_TARGET_DIR = Path("data/DataForReputation")

COMMIT_MESSAGE = "Update dissertation reputation research data"

# Git identity used for the commit
GIT_USER_NAME = "alinaantonov"
GIT_USER_EMAIL = "alina_antonov@yahoo.com"

# IMPORTANT: change to True only after reviewing the data for public release.
PUBLIC_DATA_CONFIRMED = True

# GitHub blocks normal Git files larger than 100 MiB.
MAX_GITHUB_FILE_BYTES = 100 * 1024 * 1024

# Temporary Colab working directory. This disappears when the runtime resets.
LOCAL_REPO_DIR = Path("/content/github_dissertation_repo")



In [18]:
# ============================================================
# HELPER FUNCTIONS - run_git
# ============================================================

def run_git(args, cwd=None, env=None, capture_output=False):
    """Run a git command and raise an error if it fails."""
    cmd = ["git"] + list(args)
    try:
        result = subprocess.run(
            cmd,
            cwd=cwd,
            env=env,
            text=True,
            check=True,
            capture_output=capture_output,
        )
    except subprocess.CalledProcessError as e:
        print(f"Git command failed: {' '.join(cmd)}")
        if e.stdout:
            print(f"Stdout:\n{e.stdout}")
        if e.stderr:
            print(f"Stderr:\n{e.stderr}")
        raise # Re-raise the original exception
    return result

In [19]:
# ============================================================
# HELPER FUNCTIONS - find_oversized_files
# ============================================================

def find_oversized_files(folder: Path):
    """Return files at or above GitHub's normal 100 MiB Git limit."""
    oversized = []
    for path in folder.rglob("*"):
        if path.is_file():
            try:
                size = path.stat().st_size
            except OSError:
                continue
            if size >= MAX_GITHUB_FILE_BYTES:
                oversized.append((path, size))
    return oversized


In [20]:
# ============================================================
# HELPER FUNCTIONS - make_git_auth_environment
# ============================================================
def make_git_auth_environment(token: str):
    """
    Create a temporary GIT_ASKPASS helper.

    The token is supplied through an environment variable instead of being
    stored in the repository's remote URL.
    """
    askpass = Path("/tmp/github_askpass.sh")
    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
  *) echo "$GITHUB_TOKEN" ;;
esac
"""
    )
    askpass.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)

    env = os.environ.copy()
    env["GITHUB_TOKEN"] = token
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"

    return env, askpass

In [22]:
# ============================================================
# MAIN WORKFLOW
# ============================================================

def main():
    print("1. Mounting Google Drive...")
    drive.mount("/content/drive")

    #pdb.set_trace()

    if GITHUB_USERNAME == "YOUR_GITHUB_USERNAME":
        raise ValueError("Set GITHUB_USERNAME at the top of the script.")

    if GITHUB_REPOSITORY == "YOUR_REPOSITORY_NAME":
        raise ValueError("Set GITHUB_REPOSITORY at the top of the script.")

    if GIT_USER_EMAIL == "YOUR_GITHUB_EMAIL":
        raise ValueError("Set GIT_USER_EMAIL at the top of the script.")

    if not PUBLIC_DATA_CONFIRMED:
        raise RuntimeError(
            "PUBLIC_DATA_CONFIRMED is False.\n"
            "Review DataForReputation for API keys, credentials, private/restricted "
            "data, licensing restrictions, and sensitive files. Then set "
            "PUBLIC_DATA_CONFIRMED = True."
        )

    if not SOURCE_DIR.exists():
        raise FileNotFoundError(f"Source folder not found in SOURCE_DIR: {SOURCE_DIR}")

    print(f"2. Checking source folder in SOURCE_DIR:\n   {SOURCE_DIR}")

    oversized = find_oversized_files(SOURCE_DIR)
    if oversized:
        print("\nFiles at or above 100 MiB were found:")
        for path, size in oversized:
            print(f" - {path} ({size / (1024**2):.1f} MiB)")
        raise RuntimeError(
            "Upload stopped. Standard GitHub Git pushes reject individual files "
            "at or above 100 MiB. Remove them, store them elsewhere, or configure Git LFS."
        )

    print("3. Reading GITHUB_TOKEN from Colab Secrets...")
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception as exc:
        raise RuntimeError(
            "Could not read Colab secret GITHUB_TOKEN. "
            "Add it in Colab's Secrets panel and enable notebook access."
        ) from exc

    if not token:
        raise RuntimeError("GITHUB_TOKEN is empty.")

    auth_env, askpass_path = make_git_auth_environment(token)

    repo_url = (
        f"https://github.com/{GITHUB_USERNAME}/{GITHUB_REPOSITORY}.git"
    )

    print("repo_url..." + repo_url)

    try:
        print("4. Cloning the current GitHub repository...")
        if LOCAL_REPO_DIR.exists():
            shutil.rmtree(LOCAL_REPO_DIR)

        run_git(
            ["clone", "--branch", BRANCH, repo_url, str(LOCAL_REPO_DIR)],
            env=auth_env,
        )

        print("5. Configuring Git identity...")
        run_git(
            ["config", "user.name", GIT_USER_NAME],
            cwd=LOCAL_REPO_DIR,
        )
        run_git(
            ["config", "user.email", GIT_USER_EMAIL],
            cwd=LOCAL_REPO_DIR,
        )

        destination = LOCAL_REPO_DIR / GITHUB_TARGET_DIR

        print(f"6. Synchronizing data to:\n   {GITHUB_TARGET_DIR}")

        # Remove the prior GitHub copy first so files deleted from Drive
        # are also deleted from the GitHub version.
        if destination.exists():
            shutil.rmtree(destination)

        destination.parent.mkdir(parents=True, exist_ok=True)

        shutil.copytree(
            SOURCE_DIR,
            destination,
            ignore=shutil.ignore_patterns(
                ".DS_Store",
                "Thumbs.db",
                "__pycache__",
                "*.pyc",
                ".ipynb_checkpoints",
            ),
        )

        print("7. Staging changes...")
        run_git(
            ["add", "-A", "--", str(GITHUB_TARGET_DIR)],
            cwd=LOCAL_REPO_DIR,
        )

        status = run_git(
            ["status", "--short"],
            cwd=LOCAL_REPO_DIR,
            capture_output=True,
        ).stdout.strip()

        if not status:
            print("\nNo differences found. GitHub is already synchronized.")
            return

        print("\nChanges to be committed:")
        print(status)

        print("\n8. Creating Git commit...")
        run_git(
            ["commit", "-m", COMMIT_MESSAGE],
            cwd=LOCAL_REPO_DIR,
        )

        print("9. Pushing to GitHub...")
        run_git(
            ["push", "origin", BRANCH],
            cwd=LOCAL_REPO_DIR,
            env=auth_env,
            capture_output=True,
        )

        print("\nSUCCESS")
        print(
            f"Data synchronized to:\n"
            f"https://github.com/{GITHUB_USERNAME}/{GITHUB_REPOSITORY}"
            f"/tree/{BRANCH}/{GITHUB_TARGET_DIR.as_posix()}"
        )

    finally:
        # Remove the temporary authentication helper and token from our env copy.
        if askpass_path.exists():
            askpass_path.unlink()
        auth_env.pop("GITHUB_TOKEN", None)


if __name__ == "__main__":
    main()


1. Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2. Checking source folder in SOURCE_DIR:
   /content/drive/MyDrive/ColabNotebooks/DataForReputation
3. Reading GITHUB_TOKEN from Colab Secrets...
repo_url...https://github.com/alinaantonov/DataForReputation.git
4. Cloning the current GitHub repository...
5. Configuring Git identity...
6. Synchronizing data to:
   data/DataForReputation
7. Staging changes...

Changes to be committed:
A  data/DataForReputation/AAPL_Apple_Inc_2026-07-16_to_2026-08-11_gdelt_articles.csv
A  data/DataForReputation/AAPL_company_metadata.csv
A  data/DataForReputation/AAPL_stock_data_20260716_20260811.csv
A  data/DataForReputation/AAPL_stock_data_20260716_20260811.xlsx
A  data/DataForReputation/Code/GDELTData.ipynb
A  data/DataForReputation/Code/GoogleTrends.ipynb
A  data/DataForReputation/Code/RedditData.ipynb
A  data/DataForReputation/Code/SecEdgarData.ip

CalledProcessError: Command '['git', 'push', 'origin', 'main']' returned non-zero exit status 1.

In [23]:
import subprocess
from google.colab import userdata
import os
from pathlib import Path
import stat

token = userdata.get("GITHUB_TOKEN")

askpass = Path("/tmp/github_askpass_debug.sh")
askpass.write_text("""#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
""")
askpass.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)

env = os.environ.copy()
env["GITHUB_TOKEN"] = token
env["GIT_ASKPASS"] = str(askpass)
env["GIT_TERMINAL_PROMPT"] = "0"

result = subprocess.run(
    ["git", "push", "origin", "main"],
    cwd="/content/github_dissertation_repo",
    env=env,
    text=True,
    capture_output=True
)

print("RETURN CODE:", result.returncode)
print("\nSTDOUT:")
print(result.stdout)

print("\nGITHUB ERROR:")
print(result.stderr)

RETURN CODE: 1

STDOUT:


GITHUB ERROR:
remote: error: GH013: Repository rule violations found for refs/heads/main.        
remote: 
remote: - GITHUB PUSH PROTECTION        
remote:   —————————————————————————————————————————        
remote:     Resolve the following violations before pushing again        
remote: 
remote:     - Push cannot contain secrets        
remote: 
remote:             
remote:      (?) Learn how to resolve a blocked push        
remote:      https://docs.github.com/code-security/secret-scanning/working-with-secret-scanning-and-push-protection/working-with-push-protection-from-the-command-line#resolving-a-blocked-push        
remote:             
remote:             
remote:       —— GitHub Personal Access Token ——————————————————————        
remote:        locations:        
remote:          - commit: ca079148b906f40721cad4c29b02791f3dc5adb4        
remote:            path: data/DataForReputation/SyncWithGithub.ipynb:1        
remote:             
remote:      

### Add `SyncWithGithub.ipynb` to `.gitignore`

This will tell Git to ignore the `SyncWithGithub.ipynb` file, preventing it from being committed and pushed, thus resolving the GitHub Push Protection error.

In [ ]:
# Define the path to the .gitignore file inside the local repository
gitignore_path = LOCAL_REPO_DIR / ".gitignore"

# The path to ignore, relative to the repository root
file_to_ignore = "data/DataForReputation/SyncWithGithub.ipynb"

print(f"Adding '{file_to_ignore}' to {gitignore_path}...")

# Read existing .gitignore content, if any, to avoid duplicates
existing_content = ""
if gitignore_path.exists():
    with open(gitignore_path, 'r') as f:
        existing_content = f.read()

# Check if the file is already listed in .gitignore
if file_to_ignore not in existing_content:
    with open(gitignore_path, 'a') as f:
        f.write(f"\n{file_to_ignore}\n")
    print("File added to .gitignore.")
else:
    print("File already exists in .gitignore.")

# Now, stage, commit, and push the .gitignore change
# We'll use the same run_git function from the script above

print("Staging .gitignore changes...")
run_git(
    ["add", ".gitignore"],
    cwd=LOCAL_REPO_DIR,
)

print("Committing .gitignore changes...")
run_git(
    ["commit", "-m", "Add SyncWithGithub.ipynb to .gitignore"],
    cwd=LOCAL_REPO_DIR,
)

print("Pushing .gitignore changes to GitHub...")
# Use the same authentication environment from the main function
auth_env, _ = make_git_auth_environment(userdata.get("GITHUB_TOKEN"))
run_git(
    ["push", "origin", BRANCH],
    cwd=LOCAL_REPO_DIR,
    env=auth_env,
    capture_output=True,
)

print("Successfully updated .gitignore and pushed to GitHub.")
print("You can now run the main data synchronization script again without this specific file being pushed.")

Adding 'data/DataForReputation/SyncWithGithub.ipynb' to /content/github_dissertation_repo/.gitignore...
File added to .gitignore.
Staging .gitignore changes...
Committing .gitignore changes...
Pushing .gitignore changes to GitHub...
Git command failed: git push origin main
Stderr:
remote: error: GH013: Repository rule violations found for refs/heads/main.        
remote: 
remote: - GITHUB PUSH PROTECTION        
remote:   —————————————————————————————————————————        
remote:     Resolve the following violations before pushing again        
remote: 
remote:     - Push cannot contain secrets        
remote: 
remote:             
remote:      (?) Learn how to resolve a blocked push        
remote:      https://docs.github.com/code-security/secret-scanning/working-with-secret-scanning-and-push-protection/working-with-push-protection-from-the-command-line#resolving-a-blocked-push        
remote:             
remote:             
remote:       —— GitHub Personal Access Token ————————————

CalledProcessError: Command '['git', 'push', 'origin', 'main']' returned non-zero exit status 1.